In [1]:
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('C:/Users/user/Documents/Walmart_Sales.csv')
df.head()


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [3]:
df.drop('Date',axis=1,inplace=True)

In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

# =========================
# FEATURE ENGINEERING
# =========================



# =========================
# COLUMN TYPES
# =========================
num_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment','Store', 'Holiday_Flag','Weekly_Sales']


# =========================
# PIPELINES
# =========================

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols)
])


In [5]:
X= df.drop('Weekly_Sales',axis=1)
y=df['Weekly_Sales']

In [6]:

       models={  'randomforest':{'model':RandomForestRegressor(),
                        'params':{'n_estimators':[50, 100, 200],'max_depth':[None,10, 30],'min_samples_split':[2,5,10]}},
          'svm':{'model':SVR(),
                'params':{'C':[0.1,1,10],'kernel':['linear','rbf']}}
         }
        

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
results = []

for model_name , model_parameter in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model_parameter['model'])])
    clf = GridSearchCV(estimator=model_parameter['model'], param_grid=model_parameter['params'],cv=5,scoring='r2',n_jobs=-1)
    clf.fit(X_train, y_train)
    results.append({'model':model_name,'best_score':clf.best_score_,'best_params':clf.best_params_})

In [9]:
results

[{'model': 'randomforest',
  'best_score': np.float64(0.929795787904989),
  'best_params': {'max_depth': None,
   'min_samples_split': 10,
   'n_estimators': 100}},
 {'model': 'svm',
  'best_score': np.float64(0.1173261170224342),
  'best_params': {'C': 10, 'kernel': 'linear'}}]